# Kaggle GPU Training - Real-Time Multimodal Incident Intelligence System

This notebook trains the anomaly pipeline and exports model artifacts for reuse in the project backend.

## Dataset Choice

Primary dataset source is aligned with the Kaggle notebook input page: [odins0n/video-anomaly-detection/input](https://www.kaggle.com/code/odins0n/video-anomaly-detection/input).

This notebook auto-detects the best matching UCF-Crime dataset directory under `/kaggle/input`.

Why this one:
- Closely matches surveillance incident detection tasks.
- Includes public-safety classes such as RoadAccidents, Robbery, Fighting, Assault, etc.
- Manageable size for Kaggle sessions compared with full raw-video mirrors.

In [ ]:
import torch

gpu_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if gpu_available else 0
print('CUDA available:', gpu_available)
print('Visible GPU count:', gpu_count)
if gpu_available:
    for i in range(gpu_count):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))
else:
    print('Training will run on CPU')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/saksham-1304/VIGIL-AI-Visual-Intelligence-Global-Incident-Learning'
REPO_BRANCH = 'main'
REPO_DIR = Path('/kaggle/working/incident-intel-repo')

def run_cmd(cmd):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if not REPO_DIR.exists():
    run_cmd(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    os.chdir(REPO_DIR)
    run_cmd(['git', 'fetch', 'origin', REPO_BRANCH])
    run_cmd(['git', 'reset', '--hard', f'origin/{REPO_BRANCH}'])

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
run_cmd(['git', 'rev-parse', '--short', 'HEAD'])

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'ml/requirements.txt'], check=True)

In [ ]:
from pathlib import Path
import subprocess
import sys

# Kaggle datasets are mounted under /kaggle/input (sometimes nested under /kaggle/input/datasets/*)
ROOT_CANDIDATES = [
    Path('/kaggle/input'),
    Path('/kaggle/input/datasets'),
    Path('/kaggle/input/datasets/odins0n'),
]
OUTPUT_DIR = Path('/kaggle/working/incident-intel-output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Toggle settings
USE_YOLO = True
RUN_QUALITY_EVIDENCE = True
RETRY_EVAL_ONLY = True  # Set True after a failed evaluation to skip extraction+training

preferred_names = [
    'ucf-crime-dataset',
    'crimeucfdataset',
    'ucf-crime-full',
    'ucf-crimes',
    'crimeucf',
]

def has_train_test_layout(path: Path) -> bool:
    return (path / 'Train').exists() and (path / 'Test').exists()

def detect_dataset_dir() -> Path:
    existing_roots = [root for root in ROOT_CANDIDATES if root.exists()]
    if not existing_roots:
        raise FileNotFoundError('Kaggle input mount not found. Add dataset in notebook settings.')

    # Case 1: mounted directly as /kaggle/input/Train and /kaggle/input/Test
    for root in existing_roots:
        if has_train_test_layout(root):
            return root

    # Case 2: mounted as /kaggle/input/<dataset-slug>
    for root in existing_roots:
        for name in preferred_names:
            candidate = root / name
            if candidate.exists() and candidate.is_dir():
                return candidate

    # Case 3: unknown slug containing UCF/Crime tokens
    dynamic_candidates = []
    for root in existing_roots:
        dynamic_candidates.extend(
            [
                p for p in root.iterdir()
                if p.is_dir() and ('ucf' in p.name.lower() or 'crime' in p.name.lower())
            ]
        )
    dynamic_candidates = sorted(dynamic_candidates, key=lambda p: str(p).lower())
    if dynamic_candidates:
        return dynamic_candidates[0]

    available = []
    for root in existing_roots:
        available.extend([str(p) for p in root.iterdir() if p.is_dir()])
    raise FileNotFoundError(
        'No UCF-Crime dataset folder found. '
        f'Available folders: {sorted(available)}. '
        'Add dataset odins0n/ucf-crime-dataset in Notebook Settings > Add data.'
    )

def get_supported_flags() -> dict[str, bool]:
    help_result = subprocess.run(
        [sys.executable, 'scripts/kaggle_train.py', '-h'],
        capture_output=True,
        text=True,
        check=True,
    )
    help_text = help_result.stdout
    return {
        '--amp': '--amp' in help_text,
        '--multi-gpu': '--multi-gpu' in help_text,
        '--checkpoint-dir': '--checkpoint-dir' in help_text,
        '--checkpoint-interval': '--checkpoint-interval' in help_text,
        '--num-workers': '--num-workers' in help_text,
        '--heartbeat-seconds': '--heartbeat-seconds' in help_text,
        '--progress-every': '--progress-every' in help_text,
        '--disable-yolo': '--disable-yolo' in help_text,
        '--yolo-device': '--yolo-device' in help_text,
        '--yolo-conf': '--yolo-conf' in help_text,
        '--cross-scene-min-rows': '--cross-scene-min-rows' in help_text,
        '--holdout-ratio': '--holdout-ratio' in help_text,
        '--baseline-max-train': '--baseline-max-train' in help_text,
        '--skip-extraction': '--skip-extraction' in help_text,
        '--skip-autoencoder': '--skip-autoencoder' in help_text,
        '--skip-iforest': '--skip-iforest' in help_text,
        '--skip-load-test': '--skip-load-test' in help_text,
        '--load-cameras': '--load-cameras' in help_text,
        '--load-frames-per-camera': '--load-frames-per-camera' in help_text,
        '--load-detector-mode': '--load-detector-mode' in help_text,
        '--skip-feedback-simulation': '--skip-feedback-simulation' in help_text,
        '--feedback-samples': '--feedback-samples' in help_text,
        '--feedback-label-noise': '--feedback-label-noise' in help_text,
        '--skip-quality-gate': '--skip-quality-gate' in help_text,
    }

DATASET_DIR = detect_dataset_dir()
print('Selected Kaggle dataset directory:', DATASET_DIR)

supported = get_supported_flags()
print('Advanced flags supported:', supported)

# Fail fast: if retry mode is requested but stage-skip flags are missing, do not waste a full rerun.
required_retry_flags = ['--skip-extraction', '--skip-autoencoder', '--skip-iforest']
missing_retry_flags = [flag for flag in required_retry_flags if not supported.get(flag, False)]
if RETRY_EVAL_ONLY and missing_retry_flags:
    raise RuntimeError(
        'RETRY_EVAL_ONLY is True but scripts/kaggle_train.py does not expose retry flags: '
        f"{missing_retry_flags}. This usually means the Kaggle repo is stale. "
        'Re-run the repo sync cell, confirm latest commit, then run this cell again.'
    )

# Research-grade run: YOLO semantic fusion + calibrated evaluation + quality evidence
cmd = [
    sys.executable, 'scripts/kaggle_train.py',
    '--input-dir', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'auto',
    '--epochs', '40',
    '--latent-dim', '64',
    '--batch-size', '96',
    '--frame-step', '1',
    '--max-images', '300000',
]

if supported['--amp']:
    cmd.append('--amp')
if supported['--multi-gpu']:
    cmd.append('--multi-gpu')
if supported['--checkpoint-dir']:
    cmd.extend(['--checkpoint-dir', str(OUTPUT_DIR / 'checkpoints' / 'autoencoder')])
if supported['--checkpoint-interval']:
    cmd.extend(['--checkpoint-interval', '1'])
if supported['--num-workers']:
    cmd.extend(['--num-workers', '2'])
if supported['--heartbeat-seconds']:
    cmd.extend(['--heartbeat-seconds', '15'])
if supported['--progress-every']:
    cmd.extend(['--progress-every', '1000'])
if supported['--yolo-device'] and USE_YOLO:
    cmd.extend(['--yolo-device', 'cuda'])
if supported['--yolo-conf'] and USE_YOLO:
    cmd.extend(['--yolo-conf', '0.35'])
if supported['--disable-yolo'] and not USE_YOLO:
    cmd.append('--disable-yolo')
if supported['--cross-scene-min-rows']:
    cmd.extend(['--cross-scene-min-rows', '20'])
if supported['--holdout-ratio']:
    cmd.extend(['--holdout-ratio', '0.20'])
if supported['--baseline-max-train']:
    cmd.extend(['--baseline-max-train', '15000'])

if RETRY_EVAL_ONLY:
    cmd.append('--skip-extraction')
    cmd.append('--skip-autoencoder')
    cmd.append('--skip-iforest')
    if supported['--skip-load-test']:
        cmd.append('--skip-load-test')
    if supported['--skip-feedback-simulation']:
        cmd.append('--skip-feedback-simulation')
    if supported['--skip-quality-gate']:
        cmd.append('--skip-quality-gate')
else:
    if RUN_QUALITY_EVIDENCE:
        if supported['--load-cameras']:
            cmd.extend(['--load-cameras', '4'])
        if supported['--load-frames-per-camera']:
            cmd.extend(['--load-frames-per-camera', '60'])
        if supported['--load-detector-mode']:
            cmd.extend(['--load-detector-mode', 'fallback'])
        if supported['--feedback-samples']:
            cmd.extend(['--feedback-samples', '1000'])
        if supported['--feedback-label-noise']:
            cmd.extend(['--feedback-label-noise', '0.03'])
    else:
        if supported['--skip-load-test']:
            cmd.append('--skip-load-test')
        if supported['--skip-feedback-simulation']:
            cmd.append('--skip-feedback-simulation')
        if supported['--skip-quality-gate']:
            cmd.append('--skip-quality-gate')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

output_dir = Path('/kaggle/working/incident-intel-output')
print('Output files:')
for p in sorted(output_dir.glob('*')):
    print('-', p.name, f'({p.stat().st_size/1024/1024:.2f} MB)')

checkpoint_dir = output_dir / 'checkpoints' / 'autoencoder'
if checkpoint_dir.exists():
    ckpts = sorted(checkpoint_dir.glob('*.pt'))
    print(f'\nCheckpoint files: {len(ckpts)}')
    if ckpts:
        print('Latest checkpoint:', ckpts[-1].name)

summary_path = output_dir / 'training_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('\nBest model:', summary.get('best_model'))
    print('Runtime:', summary.get('runtime'))
    print('Training config:', summary.get('training'))

eval_path = output_dir / 'eval_report.json'
if eval_path.exists():
    eval_report = json.loads(eval_path.read_text(encoding='utf-8'))
    best = eval_report.get('best_model', {})
    print('\nEval best model:', best.get('name'))
    print('Eval F1:', best.get('f1'))
    print('Eval PR-AUC:', best.get('pr_auc'))

load_path = output_dir / 'multi_camera_load_test.json'
if load_path.exists():
    load_report = json.loads(load_path.read_text(encoding='utf-8'))
    load_summary = load_report.get('summary', {})
    latency = load_summary.get('latency_ms', {})
    print('\nLoad throughput FPS:', load_summary.get('throughput_fps'))
    print('Load p95 latency (ms):', latency.get('p95'))
    print('Load SLO pass:', load_summary.get('slo', {}).get('pass'))

feedback_path = output_dir / 'feedback_simulation.json'
if feedback_path.exists():
    feedback = json.loads(feedback_path.read_text(encoding='utf-8'))
    improvement = feedback.get('improvement', {})
    print('\nFeedback model:', feedback.get('model'))
    print('Feedback F1 delta:', improvement.get('f1_delta'))

quality_path = output_dir / 'project_readiness.json'
if quality_path.exists():
    quality = json.loads(quality_path.read_text(encoding='utf-8'))
    scores = quality.get('scores', {})
    print('\nQuality overall:', scores.get('overall'))
    print('Quality research:', scores.get('research'))
    print('Quality practical:', scores.get('practical'))

## After Training

1. Download `/kaggle/working/incident-intel-output/incident_intel_training_outputs.zip`.
2. Keep heavy binaries out of Git commits by default.
3. Publish artifacts using GitHub Releases or DVC remote storage.